In [0]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

In [0]:
url = "/Workspace/Users/arthur_torres_camargo@hotmail.com/Machine-Learning/Appliances_Energy_Prediction/dataset/energydata_complete.csv"

In [0]:
df = pd.read_csv(url)
df.head()

- `date`: data e hora no formato ano-mês-dia hora:minuto:segundo
- `Appliances`: consumo de energia dos eletrodomésticos, em Wh
- `lights`: consumo de energia das luzes da casa, em Wh
- `T1`: temperatura na área da cozinha, em graus Celsius
- `RH_1`: umidade na área da cozinha, em %
- `T2`: temperatura na área da sala de estar, em graus Celsius
- `RH_2`: umidade na área da sala de estar, em %
- `T3`: temperatura na área da lavanderia
- `RH_3`: umidade na área da lavanderia, em %
- `T4`: temperatura no escritório, em graus Celsius
- `RH_4`: umidade no escritório, em %
- `T5`: temperatura no banheiro, em graus Celsius
- `RH_5`: umidade no banheiro, em %
- `T6`: temperatura fora do prédio (lado norte), em graus Celsius
- `RH_6`: umidade fora do prédio (lado norte), em %
- `T7`: temperatura na sala de passar roupa, em graus Celsius
- `RH_7`: umidade na sala de passar roupa, em %
- `T8`: temperatura no quarto 2 do adolescente, em graus Celsius
- `RH_8`: umidade no quarto 2 do adolescente, em %
- `T9`: temperatura no quarto dos pais, em graus Celsius
- `RH_9`: umidade no quarto dos pais, em %
- `To`: temperatura externa (da estação meteorológica de Chievres), em graus Celsius
- `Pressure`: pressão atmosférica (da estação meteorológica de Chievres), em mm Hg
- `RH_out`: umidade externa (da estação meteorológica de Chievres), em %
- `Wind speed`: velocidade do vento (da estação meteorológica de Chievres), em m/s
- `Visibility`: visibilidade (da estação meteorológica de Chievres), em km
- `Tdewpoint`: ponto de orvalho (da estação meteorológica de Chievres), em °C
- `rv1`: variável aleatória 1, sem unidade
- `rv2`: variável aleatória 2, sem unidade

### Entendendo o dataset

In [0]:
df.shape

In [0]:
df.columns

In [0]:
df.describe()

In [0]:
df.isnull().sum()

In [0]:
df.info()

Opa, vimos que a coluna 'date' está como object. Talvez possamos transformar em datetime

In [0]:
df['date'] = pd.to_datetime(df['date'])

In [0]:
df = df.set_index('date')
df.head()

In [0]:
print(df.index.is_monotonic_increasing)

### Entendendo minha variável alvo

Target: Appliances

In [0]:
target = df['Appliances']

In [0]:
df['Appliances'].describe()

In [0]:
sns.histplot(target, bins=50, kde=True)
plt.show()

In [0]:
sns.boxplot(target)
plt.show()

Ponto notados:
- Média menor que meu desvio padrão, significa que eu tenho uma alta dispersão de valores
- Histograma Assimetrico Positivo (Dados estão mais presentes em valores mais baixos, com uma cauda longa)
- Box plot mostra grande dispersão, e mostra também que muitos valores estão além dos quartis padrões

### Entendendo o padrão de consumo da minha variável 

Podemos usar algumas abordagens para verificar esse pico 
- Maiores valores
- Percentil alto (ex:. 95)
- Média + desvio (Média + 3 desvio padrões) -> a depender da distribuição

In [0]:
top_valores = df.nlargest(15, 'Appliances')['Appliances']

In [0]:
print(top_valores)

In [0]:
sns.set_theme(style="whitegrid")

for i, v in top_valores.items():
  pos = df.index.get_loc(i)
  janela = df.iloc[pos-12 : pos+13]['Appliances']
  janela_df = janela.reset_index()

  plt.figure(figsize=(20, 6))
  sns.lineplot(janela_df,
              x=janela_df.columns[0],
              y='Appliances',
              linewidth=2,
              color='steelblue')
  
  plt.scatter(i, v, color='red', s=120, zorder=5, label='pico')

  
  plt.title(
        f"Pico em {i:%Y-%m-%d %H:%M} | Appliances = {v}",
        fontsize=14
    )
  
  
  plt.xlabel("")
  plt.ylabel("Appliances")
  plt.legend()
  plt.grid(alpha=0.3)


  plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
  plt.gca().xaxis.set_major_locator(mdates.AutoDateLocator())
  plt.gcf().autofmt_xdate()

  plt.tight_layout()
  plt.show()